In [1]:
import numpy as np
import pandas as pd
from collections import deque
from dataclasses import dataclass


# ---------------------------------------------------------
# Config
# ---------------------------------------------------------
@dataclass
class PressureConfig:
    fs: int = 100                 # pressure sensor sampling rate
    num_sensors: int = 4          # 압력센서 개수

    buffer_seconds: float = 2.0
    window_ms: int = 500
    step_ms: int = 100

    smooth_window: int = 5        # moving average window

    low_thresh: float = 0.7
    high_thresh: float = 1.5

    feedback_duration_sec: float = 2.0


# ---------------------------------------------------------
# Preprocessing
# ---------------------------------------------------------
def moving_average(signal, window_size=5):
    if window_size <= 1:
        return signal

    if signal.ndim == 1:
        kernel = np.ones(window_size) / window_size
        return np.convolve(signal, kernel, mode="same")

    smoothed = np.zeros_like(signal)
    for c in range(signal.shape[1]):
        kernel = np.ones(window_size) / window_size
        smoothed[:, c] = np.convolve(signal[:, c], kernel, mode="same")

    return smoothed


def preprocess_pressure_window(raw_window, smooth_window=5):
    """
    raw_window shape:
    - single sensor: (T,)
    - multi sensor : (T, C)
    """
    raw_window = np.asarray(raw_window, dtype=np.float32)

    if raw_window.ndim == 1:
        raw_window = raw_window[:, None]

    smoothed = moving_average(raw_window, smooth_window)

    return smoothed


# ---------------------------------------------------------
# Feature extraction
# ---------------------------------------------------------
def compute_pressure_features(window):
    """
    window shape = (T, C)
    """
    mean_pressure = np.mean(window, axis=0)
    max_pressure = np.max(window, axis=0)
    min_pressure = np.min(window, axis=0)
    std_pressure = np.std(window, axis=0)
    total_pressure = np.sum(mean_pressure)

    return {
        "mean_pressure": mean_pressure,
        "max_pressure": max_pressure,
        "min_pressure": min_pressure,
        "std_pressure": std_pressure,
        "total_pressure": total_pressure,
    }


def compute_activation_ratio(current, baseline, eps=1e-8):
    return current / (baseline + eps)


def process_pressure_window(raw_window, config: PressureConfig):
    processed = preprocess_pressure_window(
        raw_window,
        smooth_window=config.smooth_window
    )

    features = compute_pressure_features(processed)

    return {
        "processed": processed,
        **features
    }


# ---------------------------------------------------------
# Windowing / Baseline
# ---------------------------------------------------------
def make_windows(signal, window_size, step_size):
    windows = []

    for s in range(0, len(signal) - window_size + 1, step_size):
        e = s + window_size
        windows.append(signal[s:e])

    return windows


def compute_baseline_stats(baseline_signal, config: PressureConfig):
    window_size = int(config.window_ms * config.fs / 1000)
    step_size = int(config.step_ms * config.fs / 1000)

    windows = make_windows(baseline_signal, window_size, step_size)

    mean_list = []
    total_list = []

    for w in windows:
        result = process_pressure_window(w, config)
        mean_list.append(result["mean_pressure"])
        total_list.append(result["total_pressure"])

    mean_arr = np.array(mean_list)      # (num_windows, num_sensors)
    total_arr = np.array(total_list)    # (num_windows,)

    baseline_stats = {
        "sensor_mean": np.mean(mean_arr, axis=0),
        "sensor_std": np.std(mean_arr, axis=0),
        "sensor_median": np.median(mean_arr, axis=0),

        "total_mean": float(np.mean(total_arr)),
        "total_std": float(np.std(total_arr)),
        "total_median": float(np.median(total_arr)),
    }

    return baseline_stats


# ---------------------------------------------------------
# Decision
# ---------------------------------------------------------
def judge_pressure_state(ratio, low_thresh=0.7, high_thresh=1.5):
    if ratio < low_thresh:
        return "underload"
    elif ratio <= high_thresh:
        return "good"
    else:
        return "overload"


def generate_feedback(state):
    if state == "underload":
        return "압력이 부족합니다. 해당 부위에 조금 더 체중/압력을 실어보세요."
    elif state == "good":
        return "좋습니다. 압력이 적절하게 유지되고 있습니다."
    elif state == "overload":
        return "압력이 과하게 들어가고 있습니다. 힘을 조금 빼고 자연스럽게 유지해보세요."
    else:
        return "상태를 판단할 수 없습니다."


# ---------------------------------------------------------
# Real-time Monitor
# ---------------------------------------------------------
class PressureMonitor:
    def __init__(self, config: PressureConfig):
        self.config = config

        self.buffer_size = int(config.buffer_seconds * config.fs)
        self.window_size = int(config.window_ms * config.fs / 1000)
        self.step_size = int(config.step_ms * config.fs / 1000)

        self.buffer = deque(maxlen=self.buffer_size)

        self.baseline_stats = None
        self.state_history = []
        self.feature_history = []

    def update_buffer(self, new_samples):
        """
        new_samples shape:
        - single sample multi sensor: (num_sensors,)
        - chunk: (T, num_sensors)
        """
        new_samples = np.asarray(new_samples, dtype=np.float32)

        if new_samples.ndim == 1:
            new_samples = new_samples[None, :]

        if new_samples.shape[1] != self.config.num_sensors:
            raise ValueError(
                f"센서 개수 불일치: input={new_samples.shape[1]}, "
                f"config.num_sensors={self.config.num_sensors}"
            )

        for row in new_samples:
            self.buffer.append(row)

    def set_baseline(self, baseline_signal):
        baseline_signal = np.asarray(baseline_signal, dtype=np.float32)

        if baseline_signal.ndim == 1:
            baseline_signal = baseline_signal[:, None]

        if baseline_signal.shape[1] != self.config.num_sensors:
            raise ValueError(
                f"baseline 센서 개수 불일치: input={baseline_signal.shape[1]}, "
                f"config.num_sensors={self.config.num_sensors}"
            )

        self.baseline_stats = compute_baseline_stats(
            baseline_signal,
            self.config
        )

    def analyze_current_window(self):
        if self.baseline_stats is None:
            raise ValueError("먼저 set_baseline()을 호출해야 합니다.")

        if len(self.buffer) < self.window_size:
            return None

        current_window = np.array(list(self.buffer)[-self.window_size:])

        result = process_pressure_window(current_window, self.config)

        sensor_mean = result["mean_pressure"]
        total_pressure = result["total_pressure"]

        sensor_ratio = compute_activation_ratio(
            sensor_mean,
            self.baseline_stats["sensor_mean"]
        )

        total_ratio = compute_activation_ratio(
            total_pressure,
            self.baseline_stats["total_mean"]
        )

        state = judge_pressure_state(
            total_ratio,
            self.config.low_thresh,
            self.config.high_thresh
        )

        feedback = generate_feedback(state)

        output = {
            "sensor_mean": sensor_mean,
            "sensor_ratio": sensor_ratio,
            "total_pressure": total_pressure,
            "total_ratio": total_ratio,
            "state": state,
            "feedback": feedback,
        }

        self.state_history.append(state)
        self.feature_history.append(output)

        return output

    def check_persistent_state(self, target_state="underload"):
        step_sec = self.step_size / self.config.fs
        required_steps = int(self.config.feedback_duration_sec / step_sec)

        if len(self.state_history) < required_steps:
            return False

        recent_states = self.state_history[-required_steps:]

        return all(s == target_state for s in recent_states)

    def summarize_session(self):
        if len(self.feature_history) == 0:
            return None

        total_ratios = [x["total_ratio"] for x in self.feature_history]
        total_pressures = [x["total_pressure"] for x in self.feature_history]
        states = [x["state"] for x in self.feature_history]

        summary = {
            "avg_total_pressure": float(np.mean(total_pressures)),
            "avg_total_ratio": float(np.mean(total_ratios)),
            "underload_ratio": states.count("underload") / len(states),
            "good_ratio": states.count("good") / len(states),
            "overload_ratio": states.count("overload") / len(states),
            "num_windows": len(states),
        }

        return summary


# ---------------------------------------------------------
# Report
# ---------------------------------------------------------
def history_to_dataframe(monitor: PressureMonitor):
    if len(monitor.feature_history) == 0:
        return None

    rows = []

    for i, item in enumerate(monitor.feature_history):
        row = {
            "window_idx": i,
            "time_sec": i * (monitor.step_size / monitor.config.fs),
            "total_pressure": item["total_pressure"],
            "total_ratio": item["total_ratio"],
            "state": item["state"],
        }

        for s in range(monitor.config.num_sensors):
            row[f"sensor{s+1}_mean"] = item["sensor_mean"][s]
            row[f"sensor{s+1}_ratio"] = item["sensor_ratio"][s]

        rows.append(row)

    return pd.DataFrame(rows)


def print_session_summary(monitor: PressureMonitor):
    summary = monitor.summarize_session()

    if summary is None:
        print("요약할 데이터가 없음")
        return

    print("\n" + "=" * 50)
    print("압력 센서 세션 요약")
    print("=" * 50)
    print(f"평균 Total Pressure : {summary['avg_total_pressure']:.4f}")
    print(f"평균 Total Ratio    : {summary['avg_total_ratio']:.2f}")
    print(f"underload 비율      : {summary['underload_ratio']:.2%}")
    print(f"good 비율           : {summary['good_ratio']:.2%}")
    print(f"overload 비율       : {summary['overload_ratio']:.2%}")
    print(f"총 window 수        : {summary['num_windows']}")


# ---------------------------------------------------------
# Dummy data
# ---------------------------------------------------------
def generate_dummy_pressure(
    duration_sec=10,
    fs=100,
    num_sensors=4,
    base_pressure=50,
    amplitude=5,
    noise_std=1.0
):
    t = np.linspace(0, duration_sec, duration_sec * fs)

    data = []

    for i in range(num_sensors):
        slow_change = amplitude * np.sin(2 * np.pi * 0.3 * t + i)
        noise = np.random.randn(len(t)) * noise_std
        sensor = base_pressure + slow_change + noise
        sensor = np.clip(sensor, 0, None)
        data.append(sensor)

    return np.stack(data, axis=1)  # (T, num_sensors)


# ---------------------------------------------------------
# Test
# ---------------------------------------------------------
def run_pressure_test(
    signal,
    config: PressureConfig,
    label="TEST",
    baseline_pressure=50,
    verbose=True
):
    print(f"\n{'='*60}")
    print(f"{label} TEST START")
    print(f"{'='*60}")

    monitor = PressureMonitor(config)

    baseline_signal = generate_dummy_pressure(
        duration_sec=10,
        fs=config.fs,
        num_sensors=config.num_sensors,
        base_pressure=baseline_pressure,
        amplitude=3,
        noise_std=1.0
    )

    monitor.set_baseline(baseline_signal)

    print("[Baseline stats]")
    print("sensor_mean:", monitor.baseline_stats["sensor_mean"])
    print("total_mean :", monitor.baseline_stats["total_mean"])

    chunk_size = config.step_size if hasattr(config, "step_size") else int(config.step_ms * config.fs / 1000)
    chunk_size = int(config.step_ms * config.fs / 1000)

    for i in range(0, len(signal), chunk_size):
        chunk = signal[i:i + chunk_size]
        monitor.update_buffer(chunk)

        result = monitor.analyze_current_window()

        if result is not None and verbose:
            print(
                f"[{i//chunk_size:03d}] "
                f"Total={result['total_pressure']:.2f} | "
                f"Ratio={result['total_ratio']:.2f} | "
                f"State={result['state']} | "
                f"Feedback={result['feedback']}"
            )

            if monitor.check_persistent_state("underload"):
                print("경고: 압력 부족 상태가 일정 시간 이상 지속 중")

            if monitor.check_persistent_state("overload"):
                print("경고: 압력 과다 상태가 일정 시간 이상 지속 중")

    print_session_summary(monitor)

    return monitor


# ---------------------------------------------------------
# Example usage
# ---------------------------------------------------------
config = PressureConfig(
    fs=100,
    num_sensors=4,
    buffer_seconds=2.0,
    window_ms=500,
    step_ms=100,
    low_thresh=0.7,
    high_thresh=1.5
)

test_under = generate_dummy_pressure(
    duration_sec=5,
    fs=config.fs,
    num_sensors=config.num_sensors,
    base_pressure=20
)

test_good = generate_dummy_pressure(
    duration_sec=5,
    fs=config.fs,
    num_sensors=config.num_sensors,
    base_pressure=50
)

test_over = generate_dummy_pressure(
    duration_sec=5,
    fs=config.fs,
    num_sensors=config.num_sensors,
    base_pressure=90
)

monitor_under = run_pressure_test(test_under, config, label="UNDER", verbose=False)
monitor_good = run_pressure_test(test_good, config, label="GOOD", verbose=False)
monitor_over = run_pressure_test(test_over, config, label="OVER", verbose=False)

df_good = history_to_dataframe(monitor_good)
print(df_good.head())


UNDER TEST START
[Baseline stats]
sensor_mean: [48.83595  48.720097 48.739365 48.824764]
total_mean : 195.1201934814453

압력 센서 세션 요약
평균 Total Pressure : 78.2127
평균 Total Ratio    : 0.40
underload 비율      : 100.00%
good 비율           : 0.00%
overload 비율       : 0.00%
총 window 수        : 46

GOOD TEST START
[Baseline stats]
sensor_mean: [48.800144 48.726803 48.69388  48.787853]
total_mean : 195.00868225097656

압력 센서 세션 요약
평균 Total Pressure : 195.3730
평균 Total Ratio    : 1.00
underload 비율      : 0.00%
good 비율           : 100.00%
overload 비율       : 0.00%
총 window 수        : 46

OVER TEST START
[Baseline stats]
sensor_mean: [48.82275  48.682423 48.71597  48.814728]
total_mean : 195.0358428955078

압력 센서 세션 요약
평균 Total Pressure : 351.5064
평균 Total Ratio    : 1.80
underload 비율      : 0.00%
good 비율           : 0.00%
overload 비율       : 100.00%
총 window 수        : 46
   window_idx  time_sec  total_pressure  total_ratio state  sensor1_mean  \
0           0       0.0      203.458633     1.043331 